# Phase 2 — FLAN-T5 Translation Fine-tuning
**Continuous Sign Language Translation** · Phase 2 of 2

Loads the frozen `MultiStreamSemanticEncoder` from Phase 1, then fine-tunes a `SignToTextModel` (encoder → attention pooling → adapter → FLAN-T5-small) for utterance-level sign-to-text translation.

> ⚠️ **Run Phase 1 first** — this notebook expects a Phase 1 checkpoint.

**Runtime:** GPU (T4 or better)


In [ ]:
# ── Clone repository and install dependencies ──────────────────────────
!git clone https://github.com/bencejdanko/continuous-sign-language-translation.git cslt
%cd cslt
!pip install -q -r requirements.txt

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Hugging Face authentication ───────────────────────────────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HF token: ")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

## Download Phase 1 checkpoint (if not local)
If you trained Phase 1 in a previous Colab session, download the encoder weights from HuggingFace.

In [ ]:
import os
from huggingface_hub import hf_hub_download

PHASE1_CKPT = "/content/phase1_ckpt"

# Download from HuggingFace if no local checkpoint exists
if not os.path.exists(os.path.join(PHASE1_CKPT, "best", "encoder.pt")):
    os.makedirs(os.path.join(PHASE1_CKPT, "best"), exist_ok=True)
    try:
        encoder_path = hf_hub_download(
            repo_id="bdanko/continuous-sign-language-translation",
            filename="best/encoder.pt",
            token=HF_TOKEN,
            local_dir=PHASE1_CKPT,
        )
        print(f"Encoder downloaded to {encoder_path} ✓")
    except Exception as e:
        print(f"Could not download Phase 1 checkpoint: {e}")
        print("Phase 2 will train encoder from scratch.")
else:
    print(f"Phase 1 checkpoint found at {PHASE1_CKPT}/best/ ✓")

## ⚙️ Configuration
Adjust these before running.

In [ ]:
from config import Phase2Config

cfg = Phase2Config(
    max_samples=100,           # Set to None for full dataset
    batch_size=8,
    epochs=3,
    warmup_epochs=1,
    phase1_ckpt=PHASE1_CKPT,
    ckpt_dir="/content/phase2_ckpt",
    upload_hf=False,
    hf_repo="bdanko/continuous-sign-language-translation",
    use_attention_pooling=True,
    use_ctc_head=False,
    seed=42,
)
print(f"Config: {cfg}")

## Training
Runs the Phase 2 fine-tuning loop with utterance-level supervision, staged unfreezing, and full evaluation (BLEU, ROUGE-L, chrF, exact match).

In [ ]:
from phase2_finetune import train_phase2

train_phase2(cfg)

## Upload to Hugging Face (optional)

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder

if cfg.upload_hf:
    create_repo(cfg.hf_repo, token=HF_TOKEN, exist_ok=True)
    upload_folder(
        folder_path=cfg.ckpt_dir,
        repo_id=cfg.hf_repo,
        token=HF_TOKEN,
        commit_message=f"Phase 2 checkpoint — {cfg.epochs} epochs",
    )
    print(f"Uploaded to https://huggingface.co/{cfg.hf_repo} ✓")
else:
    print("Skipping upload. Set cfg.upload_hf = True to upload.")

## Quick Inference Test

In [ ]:
from config import InferenceConfig
from inference import run_inference

inf_cfg = InferenceConfig(
    ckpt_dir=cfg.ckpt_dir,
    num_samples=3,
)
run_inference(inf_cfg)